# ML Mini Project – Titanic Survival Prediction (Logistic Regression)
**Problem Statement:** Use the Titanic dataset to predict whether a passenger survived, using exploratory data analysis, preprocessing and a Logistic Regression classifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv((__import__('pathlib').Path.cwd() / 'ML' / 'datasets' / 'titanic_train.csv') if (__import__('pathlib').Path.cwd() / 'ML' / 'datasets' / 'titanic_train.csv').exists() else (__import__('pathlib').Path.cwd().parent / 'datasets' / 'titanic_train.csv'))
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df['Survived'].value_counts().sort_index().plot(kind='bar', ax=axes[0, 0], title='Survived count')
df.groupby(['Sex', 'Survived']).size().unstack(fill_value=0).plot(kind='bar', ax=axes[0, 1], title='Survival by gender')
df.groupby(['Pclass', 'Survived']).size().unstack(fill_value=0).plot(kind='bar', ax=axes[1, 0], title='Survival by class')
df['Age'].hist(bins=30, ax=axes[1, 1]); axes[1, 1].set_title('Age distribution')
plt.tight_layout()
plt.show()


## Preprocessing

In [ ]:
data = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
data['Age'] = data['Age'].fillna(data['Age'].median())
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
data = pd.get_dummies(data, columns=['Embarked'], drop_first=True).astype(float)
data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
correlation = data.corr(numeric_only=True)
image = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation.columns)), labels=correlation.columns, rotation=90)
ax.set_yticks(range(len(correlation.index)), labels=correlation.index)
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features]
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
nums = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
cats = ['Sex', 'Embarked']
def preprocessing():
    return ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), nums),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), cats)
    ])


## Model

In [ ]:
logreg = Pipeline([('prepare', preprocessing()), ('model', LogisticRegression(max_iter=1000,random_state=42))])
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
print('Logistic Regression accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots()
ax.imshow(cm, cmap='Blues')
for (i, j), count in np.ndenumerate(cm): ax.text(j, i, str(count), ha='center', va='center')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()

In [ ]:
rf = Pipeline([('prepare',preprocessing()),('model',RandomForestClassifier(n_estimators=100, random_state=42))]).fit(X_train,y_train)
print('Random Forest accuracy:', accuracy_score(y_test, rf.predict(X_test)))

**Conclusion:** Gender and passenger class are the strongest predictors; females and 1st-class passengers had higher survival rates. Logistic Regression gives ~80% accuracy.